In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# 1. 数据读取
# ============================================================
import os
# 如果当前目录是 notebooks，切换到上级目录
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

DATA_DIR = Path("data")
CSV_PATH = DATA_DIR / '淘宝全品类全国数据.csv'

df = pd.read_csv(CSV_PATH)

print('✅ 数据读取成功！')
print(f'数据规模：{df.shape[0]} 行，{df.shape[1]} 列')
print('\n字段列表：', df.columns.tolist())
print('\n前3行数据：')
print(df.head(3))
print('\n数据信息：')
print(df.info())


✅ 数据读取成功！
数据规模：25000 行，15 列

字段列表： ['商品id', '一级品类', '二级品类', '商品标题', '商品价格', '省份', '商品销量', '店铺名称', '店铺标签', '先用后付', '退货宝', '风格', '面料', '版型', '适用季节']

前3行数据：
             商品id  一级品类 二级品类      商品标题    商品价格  省份      商品销量    店铺名称  店铺标签  \
0  \t446974700314  汽车用品   保养  保养2025新款  980.47  广东   500+人付款  粤优品汽车店  5年老店   
1  \t960353038337  食品生鲜   粮油    粮油官方正品  344.47  北京   100+人付款   诚信食品店  皇冠店铺   
2  \t765651339105  图书音像   教材  教材2025新款  261.81  香港  1000+人付款  港优品图书店  8年老店   

   先用后付  退货宝   风格   面料   版型 适用季节  
0   NaN  NaN  NaN  NaN  NaN  NaN  
1   NaN  NaN  NaN  NaN  NaN  NaN  
2  先用后付  NaN  NaN  NaN  NaN  NaN  

数据信息：
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 15 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   商品id    25000 non-null  str    
 1   一级品类    25000 non-null  str    
 2   二级品类    25000 non-null  str    
 3   商品标题    25000 non-null  str    
 4   商品价格    25000 non-null  float64
 5   省份      25000 non-null 

In [5]:
# ============================================================
# 2. NumPy：价格数组
# ============================================================
print('\n' + '='*50)
print('【价格分析】')
print('='*50)

prices = df['商品价格'].to_numpy()

print(f'价格均值：{prices.mean():.2f} 元')
print(f'价格中位数：{np.median(prices):.2f} 元')
print(f'最高价：{prices.max():.2f} 元')
print(f'最低价：{prices.min():.2f} 元')
print(f'价格不低于1000元的商品数：{(prices >= 1000).sum()} 条')
print(f'价格低于500元的商品数：{(prices < 500).sum()} 条')
print(f'价格在500-1000元的商品数：{((prices >= 500) & (prices < 1000)).sum()} 条')



【价格分析】
价格均值：938.26 元
价格中位数：617.37 元
最高价：5998.78 元
最低价：11.26 元
价格不低于1000元的商品数：8074 条
价格低于500元的商品数：11001 条
价格在500-1000元的商品数：5925 条


In [6]:
# ============================================================
# 3. 缺失值查看
# ============================================================
print('\n' + '='*50)
print('【缺失值统计】')
print('='*50)

print(df.isna().sum().sort_values(ascending=False))



【缺失值统计】
版型      22964
面料      22625
退货宝     22276
先用后付    21842
风格      21012
适用季节    20178
店铺标签     3741
商品销量        0
省份          0
商品价格        0
商品标题        0
二级品类        0
一级品类        0
商品id        0
店铺名称        0
dtype: int64


In [7]:
# ============================================================
# 4. 筛选：广东且价格 >= 1000元
# ============================================================
print('\n' + '='*50)
print('【筛选：广东且价格 >= 1000元】')
print('='*50)

condition = (df['省份'] == '广东') & (df['商品价格'] >= 1000)
guangdong_high_price = df.loc[condition, ['商品id', '一级品类', '商品价格', '省份', '商品销量']].sort_values('商品价格', ascending=False)

print(f'符合条件的商品数：{len(guangdong_high_price)} 条')
print('前10条（按价格从高到低）：')
print(guangdong_high_price.head(10))


【筛选：广东且价格 >= 1000元】
符合条件的商品数：714 条
前10条（按价格从高到低）：
                 商品id  一级品类     商品价格  省份      商品销量
22870  \t271359449904  数码家电  5984.69  广东  1000+人付款
12614  \t406439219572  数码家电  5950.45  广东  2000+人付款
9588   \t453132957133  数码家电  5931.16  广东    2万+人付款
4386   \t655995574060  数码家电  5831.18  广东    2万+人付款
14125  \t616431260865  数码家电  5809.16  广东    1万+人付款
22303  \t552994611034  数码家电  5782.79  广东   500+人付款
24162  \t554912900513  数码家电  5734.07  广东  2000+人付款
14661  \t770192134467  数码家电  5689.16  广东  2000+人付款
12798  \t472230254988  数码家电  5658.84  广东  2000+人付款
15182  \t801363237742  数码家电  5657.91  广东   200+人付款


In [8]:
# ============================================================
# 5. 分组统计：按一级品类汇总
# ============================================================
print('\n' + '='*50)
print('【一级品类汇总】')
print('='*50)

category_summary = (
    df.groupby('一级品类')
    .agg(
        商品数=('商品id', 'size'),
        平均价格=('商品价格', 'mean'),
        中位价格=('商品价格', 'median')
    )
    .sort_values('平均价格', ascending=False)
    .round(2)
)

print(category_summary)


【一级品类汇总】
       商品数     平均价格     中位价格
一级品类                        
数码家电  1712  3085.53  3116.12
钟表珠宝  1656  1981.20  1969.86
家居生活  1655  1527.68  1494.38
玩具乐器  1703  1259.17  1269.63
礼品鲜花  1659  1034.35  1037.23
运动户外  1684   811.42   801.66
医药健康  1670   791.81   779.60
服饰鞋包  1642   674.52   681.55
母婴用品  1685   666.88   666.25
汽车用品  1635   642.24   628.09
美妆护肤  1624   456.09   450.70
农资园艺  1654   453.70   449.61
食品生鲜  1648   259.59   260.66
宠物用品  1705   206.88   207.20
图书音像  1668   157.61   154.56


In [9]:
# 6. 保存CSV文件
print('\n' + '='*50)
print('【保存结果】')
print('='*50)

# 创建输出目录（相对路径）
OUTPUT_DIR = Path("output") / "day03_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 保存 category_summary
category_summary.to_csv(OUTPUT_DIR / 'category_summary.csv', encoding='utf-8-sig')
print(f'✅ category_summary.csv 已保存到 {OUTPUT_DIR}')

# 省份汇总
province_summary = (
    df.groupby('省份')
    .agg(
        商品数=('商品id', 'size'),
        平均价格=('商品价格', 'mean'),
        中位价格=('商品价格', 'median')
    )
    .sort_values('商品数', ascending=False)
    .round(2)
)

province_summary.to_csv(OUTPUT_DIR / 'province_summary.csv', encoding='utf-8-sig')
print(f'✅ province_summary.csv 已保存到 {OUTPUT_DIR}')


【保存结果】
✅ category_summary.csv 已保存到 output\day03_analysis
✅ province_summary.csv 已保存到 output\day03_analysis


In [10]:
# ============================================================
# 7. 结论
# ============================================================
print('\n' + '='*50)
print('【结论】')
print('='*50)
print('1. 本数据集共有 25,000 条商品记录，15 个字段。')
print('2. 商品价格均值为 938.26 元，中位数为 617.37 元，说明高价格商品拉高了均值。')
print('3. 广东商品数量最多，筛选后高价格商品数量已输出。')
print('4. 一级品类中，数码家电平均价格最高，宠物用品平均价格较低。')
print('5. 商品销量为文字分档（如"100+人付款"），不能直接做数值计算，需清洗。')


【结论】
1. 本数据集共有 25,000 条商品记录，15 个字段。
2. 商品价格均值为 938.26 元，中位数为 617.37 元，说明高价格商品拉高了均值。
3. 广东商品数量最多，筛选后高价格商品数量已输出。
4. 一级品类中，数码家电平均价格最高，宠物用品平均价格较低。
5. 商品销量为文字分档（如"100+人付款"），不能直接做数值计算，需清洗。


In [18]:
import os
# 如果当前目录是 notebooks，切换到上级目录
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

DATA_DIR = Path("data")
CSV_PATH = DATA_DIR / '淘宝全品类全国数据.csv'

print('当前工作目录：', Path.cwd())
print('数据文件存在：', CSV_PATH.exists())

df = pd.read_csv(CSV_PATH)

当前工作目录： C:\Users\21975\PyCharmMiscProject
数据文件存在： True


In [12]:
# ===== 任务1开始 =====
print(df.shape)

print("数据规模：", df.shape)
print("\n字段名：", df.columns.tolist())
print("\n前5行数据：")
print(df.head(5))
print("\n数据信息：")
print(df.info())

(25000, 15)
数据规模： (25000, 15)

字段名： ['商品id', '一级品类', '二级品类', '商品标题', '商品价格', '省份', '商品销量', '店铺名称', '店铺标签', '先用后付', '退货宝', '风格', '面料', '版型', '适用季节']

前5行数据：
             商品id  一级品类 二级品类        商品标题     商品价格  省份      商品销量       店铺名称  \
0  \t446974700314  汽车用品   保养    保养2025新款   980.47  广东   500+人付款     粤优品汽车店   
1  \t960353038337  食品生鲜   粮油      粮油官方正品   344.47  北京   100+人付款      诚信食品店   
2  \t765651339105  图书音像   教材    教材2025新款   261.81  香港  1000+人付款     港优品图书店   
3  \t614914975025  服饰鞋包   童装  童装修身2025新款   503.53  天津  2000+人付款  津优品服饰店专卖店   
4  \t757714643103  家居生活   装饰    装饰官方正品户外  1282.75  北京   500+人付款   时尚家居店旗舰店   

    店铺标签  先用后付  退货宝   风格   面料   版型 适用季节  
0   5年老店   NaN  NaN  NaN  NaN  NaN  NaN  
1   皇冠店铺   NaN  NaN  NaN  NaN  NaN  NaN  
2   8年老店  先用后付  NaN  NaN  NaN  NaN  NaN  
3    NaN   NaN  NaN  休闲风   羊毛  标准型  春秋季  
4  回头客3千   NaN  NaN  NaN  NaN  NaN  NaN  

数据信息：
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 15 columns):
 #   Column  Non-N

In [13]:
# ===== 任务2开始 =====
print("【所有字段类型】")
print(df.dtypes)

print("\n【缺失值数量（从高到低）】")
missing_count = df.isna().sum().sort_values(ascending=False)
print(missing_count)

print("\n【缺失率（百分比）】")
missing_rate = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print(missing_rate)

【所有字段类型】
商品id        str
一级品类        str
二级品类        str
商品标题        str
商品价格    float64
省份          str
商品销量        str
店铺名称        str
店铺标签        str
先用后付        str
退货宝         str
风格          str
面料          str
版型          str
适用季节        str
dtype: object

【缺失值数量（从高到低）】
版型      22964
面料      22625
退货宝     22276
先用后付    21842
风格      21012
适用季节    20178
店铺标签     3741
商品销量        0
省份          0
商品价格        0
商品标题        0
二级品类        0
一级品类        0
商品id        0
店铺名称        0
dtype: int64

【缺失率（百分比）】
版型      91.9
面料      90.5
退货宝     89.1
先用后付    87.4
风格      84.0
适用季节    80.7
店铺标签    15.0
商品销量     0.0
省份       0.0
商品价格     0.0
商品标题     0.0
二级品类     0.0
一级品类     0.0
商品id     0.0
店铺名称     0.0
dtype: float64


In [14]:
# ===== 任务3开始 =====
# 3.1 只选择"商品价格"列
price_series = df["商品价格"]
print("选择单列 '商品价格' 的类型:", type(price_series))

# 3.2 选择多列，保存为 product_view
product_view = df[["商品id", "一级品类", "商品价格", "省份", "商品销量"]]
print("\n选择多列 product_view 的类型:", type(product_view))
print("product_view 前5行:")
print(product_view.head())

# 3.3 使用 loc 取出前5行
print("\n使用 loc 取出前5行 (索引0到4) 的 '一级品类', '商品价格', '省份':")
print(df.loc[0:4, ['一级品类', '商品价格', '省份']])

# 3.4 使用 iloc 取出前5行
print("\n使用 iloc 取出前5行 (行0-4) 和 前4列 (列0-3):")
print(df.iloc[0:5, 0:4])

选择单列 '商品价格' 的类型: <class 'pandas.Series'>

选择多列 product_view 的类型: <class 'pandas.DataFrame'>
product_view 前5行:
             商品id  一级品类     商品价格  省份      商品销量
0  \t446974700314  汽车用品   980.47  广东   500+人付款
1  \t960353038337  食品生鲜   344.47  北京   100+人付款
2  \t765651339105  图书音像   261.81  香港  1000+人付款
3  \t614914975025  服饰鞋包   503.53  天津  2000+人付款
4  \t757714643103  家居生活  1282.75  北京   500+人付款

使用 loc 取出前5行 (索引0到4) 的 '一级品类', '商品价格', '省份':
   一级品类     商品价格  省份
0  汽车用品   980.47  广东
1  食品生鲜   344.47  北京
2  图书音像   261.81  香港
3  服饰鞋包   503.53  天津
4  家居生活  1282.75  北京

使用 iloc 取出前5行 (行0-4) 和 前4列 (列0-3):
             商品id  一级品类 二级品类        商品标题
0  \t446974700314  汽车用品   保养    保养2025新款
1  \t960353038337  食品生鲜   粮油      粮油官方正品
2  \t765651339105  图书音像   教材    教材2025新款
3  \t614914975025  服饰鞋包   童装  童装修身2025新款
4  \t757714643103  家居生活   装饰    装饰官方正品户外


In [15]:
# ===== 任务4开始 =====
# 4.1 筛选省份为"广东"的商品
guangdong = df[df['省份'] == '广东']
print(f"广东商品总数: {guangdong.shape[0]}")

# 4.2 筛选"广东"且商品价格不低于1000元
condition = (df['省份'] == '广东') & (df['商品价格'] >= 1000)
selected = df.loc[condition, ['商品id', '一级品类', '二级品类', '商品价格', '省份', '商品销量']]
# 4.3 按商品价格从高到低排序，输出前10条
selected_sorted = selected.sort_values(by='商品价格', ascending=False)
print("\n广东地区价格不低于1000元的商品，按价格从高到低排序 (前10条):")
print(selected_sorted.head(10))

# 4.4 筛选"浙江或江苏"的商品
zhejiang_or_jiangsu = df[(df['省份'] == '浙江') | (df['省份'] == '江苏')]
print(f"\n浙江或江苏商品数: {zhejiang_or_jiangsu.shape[0]}")

广东商品总数: 2303

广东地区价格不低于1000元的商品，按价格从高到低排序 (前10条):
                 商品id  一级品类 二级品类     商品价格  省份      商品销量
22870  \t271359449904  数码家电   手机  5984.69  广东  1000+人付款
12614  \t406439219572  数码家电   相机  5950.45  广东  2000+人付款
9588   \t453132957133  数码家电   相机  5931.16  广东    2万+人付款
4386   \t655995574060  数码家电   耳机  5831.18  广东    2万+人付款
14125  \t616431260865  数码家电   手机  5809.16  广东    1万+人付款
22303  \t552994611034  数码家电   耳机  5782.79  广东   500+人付款
24162  \t554912900513  数码家电   手机  5734.07  广东  2000+人付款
14661  \t770192134467  数码家电   相机  5689.16  广东  2000+人付款
12798  \t472230254988  数码家电   手机  5658.84  广东  2000+人付款
15182  \t801363237742  数码家电   相机  5657.91  广东   200+人付款

浙江或江苏商品数: 3356


In [16]:
# ===== 任务5开始 =====
# 5.1 商品价格的描述性统计
print("商品价格的描述性统计:")
print(df['商品价格'].describe().round(2))

# 5.2 一级品类的商品数
print("\n一级品类商品数统计:")
print(df['一级品类'].value_counts())

# 5.3 按一级品类分组汇总
category_summary = (
    df.groupby('一级品类')
    .agg(
        商品数=('商品id', 'size'),
        平均价格=('商品价格', 'mean'),
        中位价格=('商品价格', 'median')
    )
    .round(2)
)
category_summary_sorted = category_summary.sort_values(by='平均价格', ascending=False)
print("\n按一级品类分组汇总 (按平均价格降序):")
print(category_summary_sorted)

商品价格的描述性统计:
count    25000.00
mean       938.26
std       1017.92
min         11.26
25%        257.39
50%        617.37
75%       1211.89
max       5998.78
Name: 商品价格, dtype: float64

一级品类商品数统计:
一级品类
数码家电    1712
宠物用品    1705
玩具乐器    1703
母婴用品    1685
运动户外    1684
医药健康    1670
图书音像    1668
礼品鲜花    1659
钟表珠宝    1656
家居生活    1655
农资园艺    1654
食品生鲜    1648
服饰鞋包    1642
汽车用品    1635
美妆护肤    1624
Name: count, dtype: int64

按一级品类分组汇总 (按平均价格降序):
       商品数     平均价格     中位价格
一级品类                        
数码家电  1712  3085.53  3116.12
钟表珠宝  1656  1981.20  1969.86
家居生活  1655  1527.68  1494.38
玩具乐器  1703  1259.17  1269.63
礼品鲜花  1659  1034.35  1037.23
运动户外  1684   811.42   801.66
医药健康  1670   791.81   779.60
服饰鞋包  1642   674.52   681.55
母婴用品  1685   666.88   666.25
汽车用品  1635   642.24   628.09
美妆护肤  1624   456.09   450.70
农资园艺  1654   453.70   449.61
食品生鲜  1648   259.59   260.66
宠物用品  1705   206.88   207.20
图书音像  1668   157.61   154.56


In [17]:
# ===== 挑战任务开始 =====
provinces = ['广东', '江苏']
subset = df[df['省份'].isin(provinces)]

# 按省份分组统计
province_summary = (
    subset.groupby('省份')
    .agg(
        商品数=('商品id', 'size'),
        平均价格=('商品价格', 'mean'),
        中位价格=('商品价格', 'median')
    )
    .round(2)
)
print("广东和江苏的商品数据摘要:")
print(province_summary)

# 分别找出两个省份最常见的一级品类
print("\n各省最常见的一级品类:")
for province in provinces:
    top_category = (
        subset.loc[subset['省份'] == province, '一级品类']
        .value_counts()
        .head(1)
    )
    print(f"\n{province} 最常见一级品类:")
    print(top_category)

广东和江苏的商品数据摘要:
     商品数    平均价格    中位价格
省份                      
广东  2303  911.69  608.62
江苏  1763  936.48  592.41

各省最常见的一级品类:

广东 最常见一级品类:
一级品类
数码家电    168
Name: count, dtype: int64

江苏 最常见一级品类:
一级品类
图书音像    130
Name: count, dtype: int64
